# Aaron Randrianarisona - Bloc2 Programmation python pour la sécurité Projet final - Investigation d’un incident de sécurité

## Le scénario — « Opération Brume »

**Nordis Logistique** (entreprise fictive de transport : ~300 postes, 3 sites à Rouen, Lyon et Lille) a subi une attaque. Un lundi matin, plusieurs serveurs de fichiers sont devenus inaccessibles. La direction veut comprendre ce qui s’est passé, ce qui a été volé et comment éviter que cela se reproduise.

On vous remet les journaux des trois dernières semaines issus de sept sources différentes. Quelque part dans ces dizaines de milliers de lignes — noyées dans l’activité normale de l’entreprise — se trouvent les traces de l’attaque : l’entrée des attaquants, leur progression dans le réseau, le vol de données et le sabotage final.

Votre mission : reconstituer la chronologie de l’attaque, identifier les indicateurs de compromission et formuler des recommandations — uniquement à partir des données, en écrivant vos propres scripts.

### Pre sétup

In [1]:
!python -m venv .venv > nul 2>&1
!.venv\Scripts\activate #windows 
#!source .venv/bin/activate        # Linux / macOS
%pip install -r requirements.txt > nul 2>&1

Note: you may need to restart the kernel to use updated packages.


In [2]:
#!/usr/bin/env python3
"""
Script de démarrage — projet « Opération Brume ».

Charge les différentes sources avec pandas et affiche quelques repères.
À vous de l'étendre pour reconstituer l'attaque. Ce fichier ne contient
AUCUNE réponse : c'est un point de départ.

    pip install pandas
    python3 demarrage.py --donnees ./donnees
"""

import argparse
import json
from pathlib import Path

import pandas as pd


def charger(dossier: Path) -> dict[str, pd.DataFrame]:
    data: dict[str, pd.DataFrame] = {}
    data["vpn"] = pd.read_csv(dossier / "vpn_auth.csv", parse_dates=["horodatage"])
    data["proxy"] = pd.read_csv(dossier / "proxy_web.csv", parse_dates=["horodatage"])
    data["firewall"] = pd.read_csv(dossier / "firewall.csv", parse_dates=["horodatage"])
    data["mail"] = pd.read_csv(dossier / "mail_gateway.csv", parse_dates=["horodatage"])
    data["inventaire"] = pd.read_csv(dossier / "inventaire_parc.csv")
    # JSON Lines : un objet par ligne
    with open(dossier / "ad_events.jsonl", encoding="utf-8") as f:
        data["ad"] = pd.DataFrame(json.loads(l) for l in f)
    data["ad"]["horodatage"] = pd.to_datetime(data["ad"]["horodatage"])
    # JSON classique (liste d'objets)
    with open(dossier / "edr_alerts.json", encoding="utf-8") as f:
        data["edr"] = pd.DataFrame(json.load(f))
    return data


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--donnees", type=Path, default=Path("donnees"))
    args = p.parse_args()
    data = charger(args.donnees)

    print("=== Volumes ===")
    for nom, df in data.items():
        print(f"  {nom:12} {len(df):>7} lignes")

    print("\n=== Période couverte (VPN) ===")
    print(" ", data["vpn"].horodatage.min(), "->", data["vpn"].horodatage.max())

    print("\n=== Exemple : connexions VPN par résultat ===")
    print(data["vpn"].resultat.value_counts().to_string())

    print("\n=== À vous de jouer ===")
    print("  Pistes : connexions depuis l'étranger ? échecs répétés puis succès ?")
    print("           comptes créés la nuit ? gros volumes sortants ? IP inconnues ?")


#if __name__ == "__main__":
#    main()


#### Charger les données

In [3]:
path = Path("./donnees") # à remplacer par le chemin vers les données
data = charger(path)

In [4]:
data

{'vpn':                     horodatage      login        ip_source pays  \
 0    2026-08-31 07:41:07+02:00     y.roux   92.154.192.236   FR   
 1    2026-08-31 08:02:55+02:00   m.durand      88.120.7.43   FR   
 2    2026-08-31 08:15:00+02:00  d.garnier  176.150.205.132   FR   
 3    2026-08-31 08:23:49+02:00  t.garnier  176.150.143.126   FR   
 4    2026-08-31 08:24:04+02:00    l.leroy   88.120.208.104   FR   
 ...                        ...        ...              ...  ...   
 2082 2026-09-20 14:17:37+02:00   l.martin   92.154.161.167   FR   
 2083 2026-09-20 14:43:02+02:00    b.andre  176.150.127.241   FR   
 2084 2026-09-20 15:18:15+02:00   m.durand  176.150.241.159   FR   
 2085 2026-09-20 15:58:23+02:00    b.andre  176.150.152.135   FR   
 2086 2026-09-20 16:46:20+02:00   l.martin  176.150.113.137   FR   
 
      site_rattachement resultat motif  duree_min  
 0                siege   succes   NaN        106  
 1                siege   succes   NaN        244  
 2                s

In [5]:
data["proxy"].head(10)

,horodatage,login,ip_source,methode,host,url,statut,octets_envoyes,octets_recus,categorie
0,2026-08-31 07:03:12+02:00,s.garnier,PC-LIL-1161,GET,update.systeme.example,/,404,1483,12384,normal
1,2026-08-31 07:03:40+02:00,t.legrand,PC-SIE-1033,POST,www.transporteur-partenaire.example,/,200,507,9164,normal
2,2026-08-31 07:03:49+02:00,c.garnier,PC-LIL-1208,GET,intranet.nordis.example,/dossier/12,200,463,74100,normal
3,2026-08-31 07:03:57+02:00,h.bernard,PC-LYO-1227,GET,www.banque-pro.example,/recherche?q=colis,404,248,58911,normal
4,2026-08-31 07:04:19+02:00,b.mercier,PC-SIE-1089,POST,portail-rh.nordis.example,/dossier/12,304,429,58315,normal
5,2026-08-31 07:05:03+02:00,h.david,PC-SIE-1002,GET,cdn.scripts.example,/static/app.js,200,1511,54215,normal
6,2026-08-31 07:05:56+02:00,n.blanc,PC-LYO-1215,GET,erp.nordis.example,/api/data,200,447,22961,normal
7,2026-08-31 07:06:13+02:00,g.martin,PC-SIE-1087,POST,www.actualites.example,/recherche?q=colis,200,259,30844,normal
8,2026-08-31 07:07:47+02:00,n.francois,PC-SIE-1196,POST,video.formation.example,/api/data,304,294,27436,normal
9,2026-08-31 07:08:50+02:00,h.david,PC-SIE-1002,GET,www.banque-pro.example,/dossier/12,200,575,12352,normal


#### Reformatage et homogénisation

In [6]:
# formatage homogène des horodatages pour toutes les sources
types_journaux = [ "vpn", "proxy", "firewall", "mail", "ad", "edr"]
for t in types_journaux:
    data[t]["horodatage"] = pd.to_datetime(data[t]["horodatage"], errors="coerce", utc=True)

    # homogenisation des sources qui journalisent le nom des utilisateurs vers 'login'
    if "login" not in data[t].columns and "utilisateur" in data[t].columns:
        data[t] = data[t].rename(
            columns={"utilisateur": "login"}
        )

### Niveau 1 — Comprendre les données 

Combien d’utilisateurs, de postes et de serveurs compte l’entreprise ? Sur quelle période portent les journaux ?
À quoi ressemble une journée « normale » (volumes, horaires) ? Cette référence vous servira à repérer l’anormal.


In [7]:
# le nombre d'utilisateur selon le nombre de poste dédié à une personne dans l'inventaire (sachant que 1 utilisateur = 1 poste)
masque_poste = data["inventaire"]["type"] == "poste"
print(f"Nombre de poste dans l'inventaire : {len(data["inventaire"][masque_poste])}")
print(f"Nombre d'utilisateur selon l'inventaire : {data["inventaire"]["proprietaire"].nunique()}")

Nombre de poste dans l'inventaire : 300
Nombre d'utilisateur selon l'inventaire : 300


In [8]:
# nombre de serveurs
masque_serveur = data["inventaire"]["type"] == "serveur"
print(f"Nombre de serveur dans l'inventaire : {len(data["inventaire"][masque_serveur])}")

Nombre de serveur dans l'inventaire : 8


In [9]:
# nombre d'utilisateurs uniques
print(f"Nombre d'utilisateurs unique journalisé dans le log vpn : { data["vpn"]["login"].nunique()}")
print(f"Nombre d'utilisateurs unique journalisé dans le log mail : {data["mail"]["destinataire"].nunique()}")
print(f"Nombre d'utilisateurs unique journalisé dans le log proxy : {data["proxy"]["login"].nunique()}")
print(f"Nombre d'utilisateurs unique journalisé dans le log edr : {data["edr"]["login"].nunique()}")
print(f"Nombre d'utilisateurs unique journalisé dans le log ad : {data["ad"]["login"].nunique()}")

Nombre d'utilisateurs unique journalisé dans le log vpn : 113
Nombre d'utilisateurs unique journalisé dans le log mail : 299
Nombre d'utilisateurs unique journalisé dans le log proxy : 295
Nombre d'utilisateurs unique journalisé dans le log edr : 149
Nombre d'utilisateurs unique journalisé dans le log ad : 302


Tous les utilisateurs n'utilisent pas nécessairement les mêmes services, ni avec la même fréquence. Il est donc normal d'observer certaines différences dans le nombre d'utilisateurs recensés par chaque journal.

**Cependant**, plusieurs écarts méritent une attention particulière. Le journal de la passerelle de messagerie (mail) recense un utilisateur de moins que l'inventaire des collaborateurs (299 contre 300). À l'inverse, le journal d'authentification Windows (ad) fait apparaître davantage d'utilisateurs que ceux recensés dans l'inventaire (302 contre 300)

In [10]:
# les difference entre les utilisateurs recensé et ceux enregistré dans le log mail
users_not_in_mail = set(data["inventaire"]["proprietaire"].dropna().unique()) - set(data["mail"]["destinataire"].dropna().str.split("@").str[0].unique())
print("Utilisateurs n'étant pas journalisés dans le log de la passerelle de messagerie: ", users_not_in_mail)

# la difference entre les utilisateurs recensé et ceux enregistré dans le log ad
unkown_ad_user = set(data["ad"]["login"].dropna().unique()) - set(data["inventaire"]["proprietaire"].dropna().unique())
print("Utilisateurs journalisés dans l'AD ne figurant pas dans l'inventaire:", unkown_ad_user)

Utilisateurs n'étant pas journalisés dans le log de la passerelle de messagerie:  {'k.perrin'}
Utilisateurs journalisés dans l'AD ne figurant pas dans l'inventaire: {'j.admin_rh', 'svc_monitoring'}


In [11]:
from typing import cast

user = cast(str,next(iter(users_not_in_mail)))
print(f"===Activité de {user}===")
for t in types_journaux:
    if "login" in data[t].columns:
        masque = data[t]["login"] == user
    elif "destinataire" in data[t].columns:  # adresses e-mail du journal mail
        masque = data[t]["destinataire"].astype(str).str.split("@").str[0] == user
    else:
        print(f"{t} - aucun champ utilisateur disponible")
        continue

    print(f"{t} - {len(data[t][masque])} activité(s) - Première activité : {data[t][masque]["horodatage"].min()} -> Dernière activité : {data[t][masque]["horodatage"].max()} ")


===Activité de k.perrin===
vpn - 0 activité(s) - Première activité : NaT -> Dernière activité : NaT 
proxy - 85 activité(s) - Première activité : 2026-08-31 06:00:03+00:00 -> Dernière activité : 2026-09-17 16:56:19+00:00 
firewall - aucun champ utilisateur disponible
mail - 0 activité(s) - Première activité : NaT -> Dernière activité : NaT 
ad - 33 activité(s) - Première activité : 2026-08-31 08:18:22+00:00 -> Dernière activité : 2026-09-19 14:13:39+00:00 
edr - 1 activité(s) - Première activité : 2026-09-19 07:56:28+00:00 -> Dernière activité : 2026-09-19 07:56:28+00:00 


In [12]:
# date de début des logs & date de fin des logs
print("=== Période couverte par chaque type de journal ===")
for t in types_journaux:
    print(f"{t:10} : {data[t].horodatage.min()} -> {data[t].horodatage.max()}")
print("\n=== Periode global pour tous les journaux ===")
print(f"{min(data[t].horodatage.min() for t in types_journaux)} -> {max(data[t].horodatage.max() for t in types_journaux)}")


=== Période couverte par chaque type de journal ===
vpn        : 2026-08-31 05:41:07+00:00 -> 2026-09-20 14:46:20+00:00
proxy      : 2026-08-31 05:03:12+00:00 -> 2026-09-20 15:55:51+00:00
firewall   : 2026-08-30 22:13:11+00:00 -> 2026-09-20 21:57:21+00:00
mail       : 2026-08-31 05:09:05+00:00 -> 2026-09-18 16:27:36+00:00
ad         : 2026-08-31 05:00:10+00:00 -> 2026-09-20 16:33:01+00:00
edr        : 2026-08-31 07:26:36+00:00 -> 2026-09-20 12:28:45+00:00

=== Periode global pour tous les journaux ===
2026-08-30 22:13:11+00:00 -> 2026-09-20 21:57:21+00:00


Soit sur une periode de ~= 22 jours à partir du 30 aout à 22:13

In [13]:
from bisect import insort
from collections import defaultdict


# etablissement d'une journée type

#--- on prend sur n premiers jours 
jour_type_user: defaultdict[str, defaultdict[str, list[pd.Timestamp]]] = ( defaultdict(lambda: defaultdict(list)))
n_jours = 10
sorted_log = data["ad"].sort_values(
    by="horodatage",
    ascending=True
)
dates = sorted_log["horodatage"].dropna().dt.strftime("%Y-%m-%d").unique()[:n_jours]

for log_name in ["ad", "vpn", "proxy", "mail", "edr"]:
    masque = data[log_name]["horodatage"].dt.strftime("%Y-%m-%d").isin(dates)
    short_log = data[log_name][masque]
    for row in short_log.itertuples():
        horodatage = cast(pd.Timestamp, row.horodatage)
        date =  horodatage.strftime("%Y-%m-%d")
        if date not in jour_type_user.keys():
            jour_type_user[date] = defaultdict(list)
        login = ""
        if not hasattr(row, "login") and hasattr(row, "destinataire"): # cas particulier de mail
            login = str(row.destinataire).split("@")[0]
        elif hasattr(row, "login"):
            login = cast(str,row.login)
        if not jour_type_user[date][login]:
            jour_type_user[date][login] = list()
        insort(jour_type_user[date][login], horodatage) # au fur et a mesure qu'on parcourt les log, on établie une liste chronologique des mentions de l'utilisateur

print(f"Dates retenus: ",list(jour_type_user.keys()))
print(f"Utilisateurs présent au jour du '2026-08-31':\n",list(jour_type_user['2026-08-31'].keys()))
print(f"Activités de m.durand le '2026-08-31': ",jour_type_user['2026-08-31']['m.durand'])

Dates retenus:  ['2026-08-31', '2026-09-01', '2026-09-02', '2026-09-03', '2026-09-04', '2026-09-05', '2026-09-06', '2026-09-07', '2026-09-08', '2026-09-09']
Utilisateurs présent au jour du '2026-08-31':
 ['m.dubois', 'm.durand', 'n.morel', 'h.legrand', 'k.mercier', 'e.guerin', 'h.andre', 'b.moreau', 'c.garnier', 'p.laurent', 'g.bernard', 'j.dubois', 'i.francois', 'h.petit', 't.vincent', 'e.chevalier', 'n.gauthier', 'd.perrin', 'l.morel', 'j.roux', 's.boyer', 't.roux', 'n.francois', 'h.simon', 'h.bernard', 'd.boyer', 'p.dubois', 't.fournier', 'y.morel', 'e.moreau', 'm.vincent', 'g.chevalier', 't.garnier', 'e.bernard', 'j.guerin', 'i.moreau', 'f.dubois', 'i.gauthier', 't.gauthier', 'a.petit', 'a.david', 'h.mercier', 'n.simon', 'e.dubois', 'j.mercier', 'a.fournier', 'g.david', 'k.simon', 'n.perrin', 'j.fournier', 'n.bernard', 'e.mercier', 'j.girard', 'j.david', 'd.garnier', 'm.michel', 'i.martin', 'f.vincent', 'c.david', 'n.fournier', 'i.andre', 'i.mercier', 't.bertrand', 'h.david', 'p.be

In [ ]:
from datetime import time

# journée type moyenne
jour_arrivees: list[time] = []
jour_departs: list[time] = []

for date in jour_type_user:
    for user in jour_type_user[date]:
        min_arrivee = min(jour_type_user[date][user]).time()
        jour_arrivees.append(min_arrivee)
        max_depart = max(jour_type_user[date][user]).time()
        jour_departs.append(max_depart)

min_arrivee_global = min(jour_arrivees)
max_depart_global = max(jour_departs)

print(f"l'heure d'activité en poste minimum recensé est à, {min_arrivee_global}")
print(f"l'heure du dernier activité en poste maximum recensé est à, {max_depart_global}")

l'heure d'activité en poste minimum recensé est à, 05:00:08
l'heure du dernier activité en poste maximum recensé est à, 16:59:52


#### Synthèse niveau 1

L'étude des différents journaux a permis d'établir un périmètre global du système d'information. En s'appuyant sur les données de l'inventaire, il est notamment possible de confirmer la présence de 300 utilisateurs, chacun associé à un poste de travail dédié.

L'analyse des journaux a toutefois mis en évidence plusieurs points de vigilance :

- Un utilisateur (k.perrin) n'apparaît pas dans le journal de la passerelle de messagerie. Bien que la période couverte par les journaux s'étende sur 22 jours, il reste possible qu'aucun trafic de messagerie n'ait été généré par cet utilisateur durant cette période. Néanmoins, sa présence dans plusieurs autres journaux atteste qu'il s'agit d'un compte actif et non d'un compte désactivé.

- Des utilisateurs supplémentaires, absents de l'inventaire, sont recensés dans le journal d'authentification Windows (AD). À ce stade de l'analyse, il pourrait s'agir de comptes techniques ou de service. Cependant, ces comptes constituent une piste d'investigation pertinente dans le cadre de la reconstitution de l'incident de sécurité.

### Niveau 2 — Trouver le point d’entrée

Quel compte utilisateur a été compromis en premier, et comment (courriel piégé, page d’hameçonnage) ?
Depuis quelle adresse IP et quel pays l’attaquant s’est-il connecté ? À quelle heure ?

In [36]:
from datetime import datetime, timedelta

# definition de compromissions
seuil_tolerance_min = (datetime.combine(datetime.today(), min_arrivee_global) - timedelta(hours=0, minutes=30)).time()
seuil_tolerance_max = (datetime.combine(datetime.today(), max_depart_global) + timedelta(hours=1, minutes=30)).time()

def est_plage_suspect(horodatage: time, seuil_min: time = time(5,0,0), seuil_max: time = time(18,0,0)) -> bool :
    return horodatage < seuil_min or horodatage > seuil_max

def est_location_suspect(location: str, location_autorise: list[str] = ['FR']) -> bool:
    return not location in location_autorise

def test_est_plage_suspect():
    assert(est_plage_suspect(time(2,30,30),seuil_tolerance_min, seuil_tolerance_max) == True)
    assert(est_plage_suspect(time(23,4,30),seuil_tolerance_min, seuil_tolerance_max) == True)
    assert(est_plage_suspect(time(5,0,0),seuil_tolerance_min, seuil_tolerance_max) == False)

def test_est_location_suspect():
    assert(est_location_suspect('BR',['FR']) == True)
    assert(est_location_suspect("US",['FR','US']) == False)
    assert(est_location_suspect("DE") == True)

test_est_plage_suspect()
test_est_location_suspect()

In [39]:
# repérer l'anomalie
masque_vpn = data["vpn"]["pays"] != "FR"
data["vpn"][masque_vpn]


,horodatage,login,ip_source,pays,site_rattachement,resultat,motif,duree_min
1345,2026-09-11 21:14:00+00:00,l.perrin,203.0.113.87,BR,siege,echec,second facteur manquant,0
1346,2026-09-11 21:15:00+00:00,l.perrin,203.0.113.87,CN,siege,echec,second facteur manquant,0
1347,2026-09-11 21:17:00+00:00,l.perrin,203.0.113.87,NG,siege,succes,NaN,214
1348,2026-09-12 00:46:00+00:00,l.perrin,203.0.113.87,NG,siege,succes,NaN,246
1402,2026-09-13 23:38:00+00:00,l.perrin,203.0.113.87,RU,siege,succes,NaN,136
2042,2026-09-19 01:21:00+00:00,l.perrin,203.0.113.87,NG,siege,succes,NaN,269


In [ ]:

masque_edr = data["edr"]["statut"] == "ouvert"
incidents_actifs = data["edr"][masque_edr].sort_values(by="horodatage",ascending=True)
incidents_actifs

,id,horodatage,machine,login,titre,gravite,statut,note_analyste
118,EDR-10203,2026-09-11 09:20:00+00:00,PC-SIE-1000,l.perrin,Page d'hameçonnage visitée,moyenne,ouvert,Connexion à un domaine récemment enregistré im...
146,EDR-10204,2026-09-14 00:57:00+00:00,SRV-FILE01,svc_monitoring,Création de compte à privilèges hors procédure,critique,ouvert,Compte svc_monitoring ajouté aux Admins du dom...
188,EDR-10205,2026-09-19 02:14:00+00:00,PC-SIE-1000,l.perrin,Exfiltration de données probable,critique,ouvert,Transfert sortant volumineux vers un hébergeur...
201,EDR-10206,2026-09-20 01:59:00+00:00,SRV-FILE01,svc_monitoring,Suppression de sauvegardes,critique,ouvert,Arrêt du service de sauvegarde puis suppressio...
202,EDR-10207,2026-09-20 03:25:00+00:00,SRV-FILE01,svc_monitoring,Comportement de rançongiciel,critique,ouvert,Chiffrement massif de fichiers sur les partage...


On regardant les incidents toujours ouverts et les connexions à distance provenant de l'étranger, des éléments coincident : même utilisateur (ou presque), même addresse IP et plusieur actions suspect sur une meme periode assez courte. Du moins on a une info crucial, la date de l'enclenchement de l'incident est une possible raison

In [51]:
incident_origin = incidents_actifs.head(1)
incident_origin

,id,horodatage,machine,login,titre,gravite,statut,note_analyste
118,EDR-10203,2026-09-11 09:20:00+00:00,PC-SIE-1000,l.perrin,Page d'hameçonnage visitée,moyenne,ouvert,Connexion à un domaine récemment enregistré im...


On a possiblement une bréche par méthode de phishing le 11 septembre 2026

In [78]:
# Scan de la passarelle de la messagerie sur le login victime et à la date d'origine de l'incident
date_incident_origin = incident_origin["horodatage"].item()
victime_incident_origin = incident_origin["login"].item()

masque_mail_incident_origin = (data["mail"]["horodatage"].dt.date == date_incident_origin.date()) & (data["mail"]["destinataire"].astype(str).str.split("@").str[0] == victime_incident_origin)
data["mail"][masque_mail_incident_origin]

,horodatage,expediteur,destinataire,sujet,piece_jointe,verdict,score_risque,lien
1151,2026-09-11 08:48:00+00:00,comptabilite@fourniseur-x.example,l.perrin@nordis.example,Facture impayée - régularisation urgente,facture_relance.pdf.htm,livre,58,https://nordis-compta-auth.example/connexion


Ce courriel semble constituer le point d'origine de l'intrusion. La cible apparente est donc le service comptabilité.
Plusieurs éléments suspects peuvent déjà être relevés :
- l'extension du fichier joint est .pdf.htm, ce qui suggère une tentative de dissimulation de l'extension réelle du fichier ;
- l'objet du message, « régularisation urgente », vise manifestement à provoquer une réaction rapide de la part du destinataire ;
- le message n'a pas été identifié comme indésirable par la passerelle de messagerie ;
- le courriel contient un lien dirigeant vers un site externe, susceptible d'être utilisé pour une opération d'hameçonnage (phishing).

In [79]:
# Verification du trafic web
masque_proxy_incident_origin = (data["proxy"]["horodatage"].dt.date == date_incident_origin.date()) & (data["proxy"]["login"] == victime_incident_origin)
data["proxy"][masque_proxy_incident_origin]

,horodatage,login,ip_source,methode,host,url,statut,octets_envoyes,octets_recus,categorie
15403,2026-09-11 09:14:00+00:00,l.perrin,PC-SIE-1000,GET,nordis-compta-auth.example,/connexion,200,600,4200,hameconnage
15406,2026-09-11 09:14:40+00:00,l.perrin,PC-SIE-1000,POST,nordis-compta-auth.example,/connexion/valider,302,820,180,hameconnage


Le trafic enregistré dans le journal du proxy confirme nos soupçons. Deux flux significatifs sont observés, dont une requête HTTP POST vers nordis-compta-auth.example/connexion/valider, accompagnée de l'envoi de 820 octets de données. Cette action est immédiatement suivie d'une redirection (HTTP 302).

Ce comportement est typique d'une page d'hameçonnage destinée à collecter des identifiants. Les éléments observés permettent de conclure que l'utilisateur a vraisemblablement transmis ses identifiants à l'attaquant, conduisant à leur compromission.

In [86]:
# Repercussion sur l'authentification
masque_vpn_incident_origin = (
    data["vpn"]["horodatage"]
    .dt.time
    .apply(
        lambda h: est_plage_suspect(
            h,
            seuil_tolerance_min,
            seuil_tolerance_max
        )
    )
) & (data["vpn"]["login"] == victime_incident_origin) & (data["vpn"]["pays"].apply(est_location_suspect))
data["vpn"][masque_vpn_incident_origin]

,horodatage,login,ip_source,pays,site_rattachement,resultat,motif,duree_min
1345,2026-09-11 21:14:00+00:00,l.perrin,203.0.113.87,BR,siege,echec,second facteur manquant,0
1346,2026-09-11 21:15:00+00:00,l.perrin,203.0.113.87,CN,siege,echec,second facteur manquant,0
1347,2026-09-11 21:17:00+00:00,l.perrin,203.0.113.87,NG,siege,succes,NaN,214
1348,2026-09-12 00:46:00+00:00,l.perrin,203.0.113.87,NG,siege,succes,NaN,246
1402,2026-09-13 23:38:00+00:00,l.perrin,203.0.113.87,RU,siege,succes,NaN,136
2042,2026-09-19 01:21:00+00:00,l.perrin,203.0.113.87,NG,siege,succes,NaN,269


On comprend désormais mieux les tentatives de connexion observées dans la soirée du 11 septembre, à partir de 21 h 14. Après avoir récupéré les identifiants de sa victime, l'intrus a essayé de se connecter au VPN de l'entreprise depuis des points de connexions au brésil (BR) et en Chine (CN) mais les deux premières tentatives semblent avoir été bloquées par le mécanisme MFA.

Un point attire toutefois l'attention : la troisième tentative depuis un point de connexion au Nigeria (NG), effectuée à 21 h 17, est un succès. L'attaquant est donc finalement parvenu à passer l'authentification multifacteur et à accéder au système avec le compte compromis.

#### Synthèse Niveau 2
**Date de l'incident :** 11 septembre 2026 <br>
**Compte compromis :** l.perrin <br>
**Déclencheur :** à 09 h 14 min 40 s, l'utilisateur est victime d'une campagne d'hameçonnage l'incitant à se connecter à un faux portail destiné au service comptabilité. Les identifiants du compte sont alors compromis. <br>
**Heure de l'intrusion :** 21 h 17 <br>
**Adresse IP de l'attaquant :** 203.0.113.87 <br>
**Localisation apparente :** Nigeria (NG) <br>

 ### Niveau 3 — Suivre la progression

Repérez le balayage interne : quelle adresse, quels ports, quand ? Pourquoi cette adresse est-elle suspecte ?
Quel compte à privilèges a été compromis, et par quelle méthode ?
Quel compte l’attaquant a-t-il créé pour se maintenir, et à quel groupe l’a-t-il ajouté ?


### Niveau 4 — Mesurer l’impact

Y a-t-il eu exfiltration ? Vers quel hôte, quel jour, et combien de données (en Go) sont sorties ?
Quelles alertes de l’EDR auraient dû déclencher une réaction, et pourquoi sont-elles passées inaperçues ?


### Niveau 5 — Conclure


Établissez la chronologie complète de l’attaque et la liste des indicateurs de compromission.
Proposez des règles de détection qui auraient permis de repérer l’attaque plus tôt, et des mesures correctives.

### **Outils**

In [ ]:
# Exporter les dépendances
%pip freeze > requirements.txt

Note: you may need to restart the kernel to use updated packages.
